# 1. Introdução

O controle de sistemas dinâmicos reais frequentemente exige que o comportamento do controlador varie de acordo com o **ponto de operação** da planta.  
Em muitas aplicações industriais, mecânicas e aeroespaciais, parâmetros como velocidade, temperatura ou carga modificam de forma significativa a dinâmica do sistema, de modo que um único controlador linear fixo tende a perder desempenho quando o sistema se afasta da condição em que foi projetado.

Historicamente, estratégias em que os parâmetros de controle variam com o ponto de operação começaram a ser exploradas em sistemas aeronáuticos e em turbinas a gás, como forma de lidar com grandes variações de regime em uma mesma malha de controle. Mais tarde, essas ideias foram formalizadas na literatura de controle por agendamento de ganhos e sistemas LPV (*Linear Parameter-Varying*), consolidando-se como uma ferramenta padrão em diversas áreas de engenharia.

Uma abordagem bastante utilizada para lidar com esse tipo de situação é o 
<span style="color:#64B5F6;font-weight:bold">controle por agendamento de ganhos</span> (*gain scheduling*).  
Nesse tipo de estratégia, os parâmetros do controlador são ajustados em função de uma **variável de agendamento** ρ, que representa o ponto de operação (por exemplo, nível de tanque, regime de carga ou faixa de velocidade).  
Quando os ganhos variam de forma **contínua e suave** com ρ, a transição entre diferentes regimes operacionais tende a ser mais estável e previsível.

Atualmente, técnicas de agendamento de ganhos aparecem em aplicações como controle de aeronaves e veículos autônomos, controle de processos químicos com variação de temperatura e vazão, sistemas de geração de energia, acionamento de motores e várias outras situações em que a dinâmica muda de forma significativa com condições de operação.

Neste projeto, denominado 
<span style="color:#64B5F6;font-weight:bold">Continuous Gain Scheduling</span>, 
será estudada e implementada uma forma de **interpolação contínua de ganhos entre pontos de operação** para sistemas não lineares SISO (*Single-Input Single-Output*).  
A ideia central é:

- linearizar o sistema em diferentes pontos de operação;
- projetar controladores locais para cada ponto;
- e, em seguida, construir uma lei de controle K(ρ) contínua a partir desses ganhos locais.

Para isso, serão utilizadas técnicas de 
<span style="color:#FFB74D;font-weight:bold">otimização convexa</span>, em particular:

- <span style="color:#FFB74D;font-weight:bold">programação quadrática (QP)</span>, na formulação de problemas de síntese de controladores locais (por exemplo, PID com restrições em atuadores);  
- <span style="color:#FFB74D;font-weight:bold">mínimos quadrados com regularização</span>, para ajustar uma curva de ganhos K(ρ) que concilie boa aproximação dos ganhos locais com suavidade em relação à variável de agendamento.

A metodologia adotada no trabalho será organizada em quatro etapas principais:

1. **Pontos de operação e linearização local**  
   Cálculo de pontos de equilíbrio (trims) e obtenção de modelos lineares (Aᵢ, Bᵢ, Cᵢ, Dᵢ) associados a diferentes valores de ρᵢ.

2. **Síntese de controladores locais**  
   Projeto de controladores LQR ou PID formulados via QP para cada modelo linearizado.

3. **Interpolação contínua de ganhos K(ρ)**  
   Ajuste de uma lei de ganhos em função de ρ por meio de um problema de mínimos quadrados regularizados.

4. **Simulação não linear e avaliação de desempenho**  
   Simulação dos modelos não lineares sob um perfil ρ(t) variável, utilização da lei K(ρ(t)) e análise do desempenho global do sistema, incluindo transições entre pontos de operação.

<span style="color:#B0BEC5"><em>Referências básicas</em></span>  
- [1] Boyd, S., & Vandenberghe, L. (2004). <em>Convex Optimization</em>. Cambridge University Press.  
- [2] Apkarian, P., Gahinet, P., & Becker, G. (1995). <em>LPV Control and Gain Scheduling</em>. IEEE Transactions on Automatic Control, 40(11), 1826–1840.  
- [3] “Least Squares (Slide 8)”, notas de aula de COS360 – Otimização (ECI) e COM361 – Introd. à Otimização (ECA), 2025/2.  
- [4] “Equality Constraints and Tradeoffs (Slide 9)”, notas de aula de COS360 – Otimização (ECI) e COM361 – Introd. à Otimização (ECA), 2025/2.  
- [5] “QPs and QCQPs (Slide 12)”, notas de aula de COS360 – Otimização (ECI) e COM361 – Introd. à Otimização (ECA), 2025/2.


# 2. Modelo matemático

Nesta seção é apresentado o modelo matemático que será usado ao longo do trabalho e como ele se conecta diretamente aos tópicos de otimização vistos em COM361.

Do ponto de vista de modelagem, o objetivo é escrever o comportamento dinâmico da planta como

- uma **equação de igualdade** que descreve a dinâmica (restrição do sistema),
- e **funções de custo quadráticas** que medem desempenho de controle.

Isso permite reformular o problema de controle em termos de **least squares** e **programação quadrática (QP)**, em linha com os Slides 8, 9 e 12.

## 2.1 Estrutura geral do sistema SISO

O foco do trabalho é em sistemas SISO (*Single-Input Single-Output*), descritos por uma dinâmica não linear

- estado: $x(t) \in \mathbb{R}^n$  
- entrada de controle: $u(t) \in \mathbb{R}$  
- saída medida: $y(t) \in \mathbb{R}$  
- variável de agendamento: $\rho \in \mathbb{R}$ (escalar), associada ao ponto de operação

e uma equação de estado genérica:

$$
\dot{x}(t) = f(x(t), u(t), \rho), \qquad y(t) = g(x(t), u(t), \rho).
$$

A variável $\rho$ representa o ponto de operação (por exemplo, nível de tanque, regime de carga ou velocidade) e será usada para construir uma lei de controle $K(\rho)$ obtida a partir de controladores projetados em diferentes valores $\rho_i$.

<span style="color:#64B5F6;font-weight:bold">Perspectiva de otimização:</span>  
essa forma de modelar o sistema é útil porque:

- torna explícitas as **restrições de igualdade** (dinâmica do sistema), que aparecem naturalmente em problemas com equações de estado e vínculos de consistência;
- fornece uma estrutura direta para formular problemas de **least squares**, quando se ajustam parâmetros de um modelo $K(\rho)$ para aproximar ganhos locais obtidos em diferentes pontos de operação;
- encaixa a dinâmica linearizada e as limitações em uma forma típica de **programação quadrática (QP)**, em que se minimiza um custo quadrático em função de $u$ ou de parâmetros do controlador, sujeito a restrições lineares.

## 2.2 Da modelagem ao problema de otimização

De forma resumida, a ligação entre o modelo e os problemas de otimização usados no projeto é:

- a dinâmica linearizada em torno de um ponto $(x^*, u^*, \rho_i)$ fornece um modelo local do tipo  
  $$
  \dot{\tilde{x}}(t) = A_i \tilde{x}(t) + B_i \tilde{u}(t),
  $$
  onde $\tilde{x}$ e $\tilde{u}$ são variáveis desviadas;

- para cada ponto $\rho_i$, o projeto de um controlador local (por exemplo, um ganho de realimentação de estados) é tratado como um problema de **otimização convexa**:  
  minimiza-se um custo quadrático (erro, esforço de controle, etc.) sujeito à dinâmica linear e, se necessário, a **restrições de igualdade e desigualdade** (como limites em $u$);

- ao reunir os ganhos locais $K_i$ obtidos em cada ponto $\rho_i$, formulamos um problema de **mínimos quadrados regularizado** para ajustar uma função $K(\rho)$ contínua, o que se encaixa diretamente na estrutura de least squares dos Slides 8 e 9.

Assim, a modelagem não linear é o ponto de partida para chegar a dois tipos de problemas de otimização centrais na disciplina:

- problemas de <span style="color:#FFB74D;font-weight:bold">least squares</span> para ajustar $K(\rho)$;  
- problemas de <span style="color:#FFB74D;font-weight:bold">QP</span> para projetar controladores locais com custo quadrático e restrições lineares.

## 2.3 Modelo de tanque de nível não linear

Como primeiro exemplo específico, considera-se um tanque com saída por gravidade.  
O estado do sistema é o nível do fluido no tanque, denotado por $h(t)$, e a entrada $u(t)$ representa a vazão de entrada (controlável). A área efetiva de saída depende da variável de agendamento $\rho$, que pode ser interpretada, por exemplo, como abertura relativa de uma válvula ou condição externa de operação.

A dinâmica é descrita por:

$$
\dot{h}(t) = -\frac{a(\rho)}{A} \sqrt{2 g\, h(t)} + \frac{1}{A} u(t),
$$

onde:

- $A$ é a área da seção transversal do tanque;  
- $g$ é a aceleração da gravidade;  
- $a(\rho)$ é a área efetiva do orifício de saída, dependente de $\rho$;  
- $h(t) \ge 0$ é o nível do fluido;  
- $u(t)$ é a vazão de entrada.

A função $a(\rho)$ será modelada como uma relação afim simples:

$$
a(\rho) = a_0 + \rho (a_1 - a_0),
$$

com $a_0$ e $a_1$ representando, por exemplo, uma abertura mínima e máxima de válvula.  
Essa escolha garante uma dependência suave e contínua em $\rho$, o que é conveniente para a etapa posterior de **interpolação contínua de ganhos**.

### 2.3.1 Pontos de operação (equilíbrio)

Para um valor fixo de $\rho$ e uma vazão de entrada constante $u^*$, o ponto de operação $(h^*, u^*)$ é definido pela condição de regime permanente:

$$
\dot{h} = 0 \quad \Longrightarrow \quad f(h^*, u^*, \rho) = 0.
$$

No caso do tanque, essa condição leva a uma expressão explícita para $h^*(u^*, \rho)$, que será usada para:

- construir uma tabela de pontos de equilíbrio $(h^*, u^*, \rho)$;  
- linearizar o sistema em torno de cada ponto, obtendo matrizes locais $(A_i, B_i)$;  
- comparar, posteriormente, o desempenho de controladores locais e da lei interpolada $K(\rho)$.

<span style="color:#B0BEC5"><em>Observação:</em> este modelo será usado como caso base para ilustrar a linearização, o projeto local de controladores e a formulação dos problemas de least squares e QP abordados ao longo do projeto.</span>
